# S6E11 GPU · CatBoost GPU + TE cho cột số

Thử nghiệm trên dữ liệu chính thức của cuộc thi. Cách chia 5 fold và seed 42 giống đợt trước để so sánh trực tiếp.

## 1. Dữ liệu và môi trường GPU

Nguồn nhãn là `train.csv` chính thức của S6E11. Mã kiểm tra SHA-256 phải khớp
bản dữ liệu của repo. Tập kiểm tra của Kaggle không có nhãn và ROC AUC trong
notebook được đo trên các giao dịch giữ lại từ tập huấn luyện có nhãn.
`id` chỉ dùng để truy vết. Mỗi mô hình xác nhận thiết bị GPU trước khi fit.

In [1]:
import hashlib
import json
import os
from pathlib import Path
from subprocess import check_output
import sys
from time import perf_counter
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
SMOKE = os.environ.get('S6E11_SMOKE') == '1'
ROOT = Path(os.environ.get(
    'S6E11_PROJECT', r'D:\UET\ML\Predicting Credit Card Fraud'
)).resolve(strict=True)
REPO = ROOT / 'INT3405E_FD'
CONFIG = json.loads((REPO / 'configs' / 'default.json').read_text(encoding='utf-8-sig'))
MANIFEST = json.loads((REPO / 'data' / 'manifest.json').read_text(encoding='utf-8-sig'))
TARGET = CONFIG['target']
NUMERIC = CONFIG['numeric_columns']
FLAGS = CONFIG['flag_columns']
CATEGORICAL = CONFIG['categorical_columns']

GPU_NAME = check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        text=True, timeout=15).splitlines()[0].strip()
if 'RTX 4060' not in GPU_NAME:
    raise RuntimeError(f'GPU hiện tại khác RTX 4060 đã kiểm tra: {GPU_NAME}')
print(f'GPU phát hiện: {GPU_NAME}')

def load_competition_train():
    expected_hash = MANIFEST['input_files']['train.csv']['sha256']
    csv_path = ROOT / 'train.csv'
    if csv_path.is_file():
        with csv_path.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv không khớp manifest')
        return pd.read_csv(csv_path), digest

    with ZipFile(ROOT / 'playground-series-s6e11.zip') as archive:
        with archive.open('train.csv') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv trong ZIP không khớp manifest')
        with archive.open('train.csv') as stream:
            return pd.read_csv(stream), digest

frame, DATA_HASH = load_competition_train()
feature_names = NUMERIC + FLAGS + CATEGORICAL
required = ['id', TARGET, *feature_names]
if (set(frame.columns) != set(required) or len(frame) != MANIFEST['train_rows']
        or frame.id.duplicated().any() or frame[required].isna().any().any()
        or set(frame[TARGET].unique()) != {0, 1}):
    raise ValueError('Dữ liệu huấn luyện không khớp schema đã kiểm tra của repo')
if SMOKE:
    _, frame = train_test_split(
        frame, test_size=20_000, stratify=frame[TARGET], random_state=SEED,
    )
    frame = frame.reset_index(drop=True)
y = frame[TARGET].to_numpy(dtype=np.uint8)
ids = frame['id'].to_numpy(dtype=np.int64)
X = frame[feature_names]
raw_numeric = X[NUMERIC + FLAGS].to_numpy(dtype=np.float32)
if not np.isfinite(raw_numeric).all():
    raise ValueError('Đặc trưng số có giá trị không hữu hạn')
print(f'Số dòng: {len(y):,} | nhãn gian lận: {y.sum():,} ({y.mean():.2%})')

ROUND2_DIR = ROOT / 's6e11_gpu' / 'round2'

GPU phát hiện: NVIDIA GeForce RTX 4060 Laptop GPU
Số dòng: 699,960 | nhãn gian lận: 11,689 (1.67%)


## 2. Năm phần chia phân tầng

Dùng `StratifiedKFold` với seed 42 như repo để giữ tỷ lệ gian lận giữa các phần.
Mỗi giao dịch được chấm một lần bởi mô hình học từ bốn phần còn lại.
Số vòng hoặc số epoch được cố định trước khi đánh giá phần giữ lại.

In [2]:
n_folds = 2 if SMOKE else 5
splitter = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
splits = list(splitter.split(X, y))
assignment = np.full(len(y), -1, dtype=np.int8)
for fold, (_, val_idx) in enumerate(splits):
    assignment[val_idx] = fold
if (assignment < 0).any():
    raise RuntimeError('Có giao dịch chưa được gán fold')
SPLIT_HASH = hashlib.sha256(ids.astype('<i8').tobytes() + assignment.tobytes()).hexdigest()
print(f'{n_folds} fold | mã kiểm tra cách chia: {SPLIT_HASH[:16]}')

5 fold | mã kiểm tra cách chia: e3629fd42119a82d


## 3. Mã hóa mục tiêu trong phần huấn luyện

Dùng 13 cột số và cách làm trơn của bước 7 trong repo. Mỗi giá trị số được đổi
thành khóa nguyên với độ chính xác hai chữ số thập phân. Mã hóa là tỷ lệ gian
lận theo khóa, được kéo về tỷ lệ chung với trọng số 20 mẫu. Khóa chưa gặp nhận
tỷ lệ chung của phần tham chiếu. Trong tập huấn luyện, mỗi dòng được mã hóa từ
bốn phần khác của một phép chia nội bộ (cross-fitting). Tập xác thực dùng
thống kê từ toàn bộ phần huấn luyện tương ứng. Đặc trưng gốc và cột phân loại
mã hóa one-hot vẫn được giữ để mô hình có thể dùng cùng với TE.

In [3]:
from sklearn.preprocessing import OneHotEncoder

te_config = CONFIG['training_plan']['target_encoding']
smoothing = float(te_config['smooth'])
inner_folds = int(te_config['inner_folds'])
key_scale = int(te_config['key_scale'])
te_seed = int(CONFIG['encoder_seed'])

# Chỉ đánh số khóa theo giá trị, chưa dùng nhãn ở bước này.
numeric_codes = []
for column in NUMERIC:
    keys = np.rint(X[column].to_numpy(dtype=np.float64) * key_scale).astype(np.int64)
    values, codes = np.unique(keys, return_inverse=True)
    numeric_codes.append((codes.astype(np.int32), len(values)))

def lookup_rates(codes, labels, n_keys):
    counts = np.bincount(codes, minlength=n_keys)
    positives = np.bincount(codes, weights=labels, minlength=n_keys)
    return ((positives + smoothing * labels.mean()) / (counts + smoothing)).astype(np.float32)

def prepare_ml_fold(train_idx, val_idx, fold):
    labels = y[train_idx]
    inner = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=te_seed)
    inner_splits = list(inner.split(train_idx, labels))
    te_train = np.empty((len(train_idx), len(NUMERIC)), dtype=np.float32)
    te_val = np.empty((len(val_idx), len(NUMERIC)), dtype=np.float32)
    for col, (codes, n_keys) in enumerate(numeric_codes):
        train_codes = codes[train_idx]
        for fit, held in inner_splits:
            rates = lookup_rates(train_codes[fit], labels[fit], n_keys)
            te_train[held, col] = rates[train_codes[held]]
        te_val[:, col] = lookup_rates(train_codes, labels, n_keys)[codes[val_idx]]

    one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32)
    categorical_train = one_hot.fit_transform(X.iloc[train_idx][CATEGORICAL])
    categorical_val = one_hot.transform(X.iloc[val_idx][CATEGORICAL])
    train_values = np.concatenate((raw_numeric[train_idx], categorical_train, te_train), axis=1)
    val_values = np.concatenate((raw_numeric[val_idx], categorical_val, te_val), axis=1)
    print(f'Fold {fold + 1}: {train_values.shape[1]} cột sau mã hóa')
    return train_values, val_values

## CatBoost trên GPU

CatBoost nhận các cột số, cột cờ và cột phân loại gốc. Mô hình dùng thêm
13 cột mã hóa theo nhãn (target encoding), tính riêng trong mỗi phần chia.
Mô hình học trong 900 vòng với cây sâu 4. Tập xác thực chỉ dùng để tạo
dự đoán. Notebook kiểm tra CatBoost đã huấn luyện trên RTX 4060.

In [4]:
import sys
from subprocess import check_output
from time import perf_counter

catboost_dependencies = ROUND2_DIR / ".build" / "deps_catboost"
if catboost_dependencies.is_dir():
    sys.path.insert(0, str(catboost_dependencies))

import catboost
from catboost import CatBoostClassifier, Pool
from catboost.utils import get_gpu_device_count

catboost_gpu_count = get_gpu_device_count()
catboost_gpu_name = check_output(
    ["nvidia-smi", "--id=0", "--query-gpu=name", "--format=csv,noheader"],
    text=True, timeout=15,
).strip()
if catboost_gpu_count != 1 or "RTX 4060" not in catboost_gpu_name:
    raise RuntimeError(
        f"CatBoost cần đúng một GPU CUDA là RTX 4060, hiện thấy "
        f"{catboost_gpu_count} GPU CUDA và {catboost_gpu_name!r} ở khe 0."
    )

### Huấn luyện từng fold

Giữ cột phân loại gốc, thêm TE cho cột số rồi chuyển dữ liệu cho CatBoost.

In [5]:
def fit_predict(train_idx, val_idx, fold):
    train_values, val_values = prepare_ml_fold(train_idx, val_idx, fold)
    te_names = [f"te_{column}" for column in NUMERIC]
    n_te = len(te_names)
    train_te = train_values[:, -n_te:].copy()
    val_te = val_values[:, -n_te:].copy()
    del train_values, val_values

    columns = [*NUMERIC, *FLAGS, *CATEGORICAL]
    train_features = X.iloc[train_idx][columns].copy()
    val_features = X.iloc[val_idx][columns].copy()
    for column in CATEGORICAL:
        train_features[column] = train_features[column].astype(str)
        val_features[column] = val_features[column].astype(str)
    train_features[te_names] = train_te
    val_features[te_names] = val_te

    train_pool = Pool(train_features, y[train_idx], cat_features=CATEGORICAL)
    val_pool = Pool(val_features, cat_features=CATEGORICAL)
    iterations = 24 if SMOKE else 900
    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=iterations,
        depth=4,
        learning_rate=0.05,
        boosting_type="Plain",
        max_ctr_complexity=1,
        task_type="GPU",
        devices="0",
        random_seed=SEED,
        use_best_model=False,
        allow_writing_files=False,
        verbose=False,
    )
    start = perf_counter()
    model.fit(train_pool)
    fit_seconds = perf_counter() - start

    params = model.get_all_params()
    if params["task_type"] != "GPU" or model.tree_count_ != iterations:
        raise RuntimeError("CatBoost không xác nhận đủ số vòng huấn luyện trên GPU.")
    device = f"CUDA 0: {catboost_gpu_name}"
    print(f"CatBoost dùng {device}, phiên bản {catboost.__version__}", flush=True)

    scores = model.predict_proba(val_pool)[:, 1]
    return np.asarray(scores).reshape(-1), {
        "device": device,
        "fit_seconds": fit_seconds,
        "version": catboost.__version__,
        "task_type": params["task_type"],
        "cuda_device": model.get_param("devices"),
        "visible_gpu_count": catboost_gpu_count,
        "iterations": model.tree_count_,
        "depth": params["depth"],
    }

## Đánh giá

ROC AUC đo chất lượng xếp hạng giao dịch gian lận. Điểm OOF gộp dùng các dự
đoán từ cả năm phần giữ lại. Average Precision (AP) tóm tắt độ chính xác của
cảnh báo khi thay đổi ngưỡng. Tỷ lệ gian lận là mốc tham khảo cho AP ngẫu nhiên.
Thời gian fit được đo trong từng mô hình. Thời gian toàn lượt còn gồm tiền xử
lý và suy luận.

In [6]:
MODEL_SLUG = 'catboost-te'
output_dir = ROUND2_DIR / ('.build/smoke_results' if SMOKE else 'results')
output_dir.mkdir(parents=True, exist_ok=True)
oof = np.full(len(y), np.nan, dtype=np.float64)
fold_metrics = []
run_start = perf_counter()
for fold, (train_idx, val_idx) in enumerate(splits[:1] if SMOKE else splits):
    print(f'Bắt đầu fold {fold + 1}/{1 if SMOKE else len(splits)}', flush=True)
    if np.isfinite(oof[val_idx]).any():
        raise RuntimeError('Fold xác thực bị trùng')
    scores, details = fit_predict(train_idx, val_idx, fold)
    scores = np.asarray(scores, dtype=np.float64)
    if (scores.shape != (len(val_idx),) or not np.isfinite(scores).all()
            or (scores < 0).any() or (scores > 1).any()):
        raise RuntimeError(f'Fold {fold + 1} trả về xác suất không hợp lệ')
    if 'RTX 4060' not in details['device']:
        raise RuntimeError(f"Fold {fold + 1} không xác minh được RTX 4060: {details['device']}")
    oof[val_idx] = scores
    fold_auc = float(roc_auc_score(y[val_idx], scores))
    record = dict(details, fold=fold + 1, validation_rows=len(val_idx), roc_auc=fold_auc)
    fold_metrics.append(record)
    print(f"Fold {fold + 1}: ROC AUC {fold_auc:.6f} | {details['fit_seconds']:.2f}s | {details['device']}", flush=True)

evaluated = np.isfinite(oof)
if not SMOKE and not evaluated.all():
    raise RuntimeError('Dự đoán OOF chưa phủ toàn bộ tập huấn luyện')
metrics = {
    'model': MODEL_SLUG,
    'status': 'completed',
    'smoke': SMOKE,
    'models_use_llm': False,
    'rows': int(evaluated.sum()),
    'positives': int(y[evaluated].sum()),
    'train_sha256': DATA_HASH,
    'split_sha256': SPLIT_HASH,
    'roc_auc': float(roc_auc_score(y[evaluated], oof[evaluated])),
    'average_precision': float(average_precision_score(y[evaluated], oof[evaluated])),
    'fit_seconds': float(sum(record['fit_seconds'] for record in fold_metrics)),
    'wall_seconds': float(perf_counter() - run_start),
    'folds': fold_metrics,
    'versions': {name: sys.modules[name].__version__
                 for name in ('numpy', 'pandas', 'sklearn', 'torch', 'lightgbm', 'xgboost')
                 if name in sys.modules},
}
result = output_dir / f'{MODEL_SLUG}.json'
temp = result.with_suffix('.tmp')
temp.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
temp.replace(result)
print(f"{'Smoke AUC' if SMOKE else 'ROC AUC OOF'}: {metrics['roc_auc']:.6f}")
print(f"AP: {metrics['average_precision']:.6f} | Fit: {metrics['fit_seconds']:.2f}s")

oof_path = output_dir / f'{MODEL_SLUG}-oof.npz'
temp_oof = oof_path.with_suffix('.tmp')
with temp_oof.open('wb') as stream:
    np.savez(stream, ids=ids[evaluated], fold=assignment[evaluated],
             prediction=oof[evaluated], train_sha256=DATA_HASH, split_sha256=SPLIT_HASH)
temp_oof.replace(oof_path)
print(f'Đã lưu dự đoán theo giao dịch: {oof_path.name}')

Bắt đầu fold 1/5
Fold 1: 67 cột sau mã hóa
CatBoost dùng CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU, phiên bản 1.2.10
Fold 1: ROC AUC 0.945438 | 28.15s | CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU
Bắt đầu fold 2/5
Fold 2: 67 cột sau mã hóa
CatBoost dùng CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU, phiên bản 1.2.10
Fold 2: ROC AUC 0.948447 | 24.07s | CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU
Bắt đầu fold 3/5
Fold 3: 67 cột sau mã hóa
CatBoost dùng CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU, phiên bản 1.2.10
Fold 3: ROC AUC 0.952827 | 28.46s | CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU
Bắt đầu fold 4/5
Fold 4: 67 cột sau mã hóa
CatBoost dùng CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU, phiên bản 1.2.10
Fold 4: ROC AUC 0.947685 | 28.21s | CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU
Bắt đầu fold 5/5
Fold 5: 67 cột sau mã hóa
CatBoost dùng CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU, phiên bản 1.2.10
Fold 5: ROC AUC 0.941646 | 29.65s | CUDA 0: NVIDIA GeForce RTX 4060 Laptop GPU
ROC AUC OOF: 0.